# Semi-Supervised Learning

## What is Semi-Supervised Learning?

Semi-supervised learning (SSL) sits between **supervised learning** (all data labeled) and **unsupervised learning** (no labels). It uses a **small amount of labeled data** combined with a **large amount of unlabeled data** for training.

### Why Does It Matter?
Labeling data is expensive and time-consuming:
- Medical images require expert radiologists
- Legal documents need domain specialists
- Speech data needs transcription

SSL allows us to exploit the abundant unlabeled data to improve generalization.

---

## Core Assumptions

### 1. Smoothness Assumption
If two points $x_1$ and $x_2$ are close in input space, their labels $y_1$ and $y_2$ should also be similar:
$$\text{If } x_1 \approx x_2 \Rightarrow y_1 \approx y_2$$

### 2. Cluster Assumption
Data points in the same cluster tend to share the same label. The decision boundary should lie in low-density regions.

### 3. Manifold Assumption
High-dimensional data lies on a lower-dimensional manifold. Learning the manifold structure from unlabeled data helps.

---

## 1. Self-Training

The simplest SSL approach:
1. Train a classifier on labeled data $\mathcal{L}$
2. Predict labels for unlabeled data $\mathcal{U}$
3. Add high-confidence predictions to $\mathcal{L}$
4. Repeat until convergence

$$\mathcal{L}_{new} = \mathcal{L} \cup \{(x, \hat{y}) : x \in \mathcal{U}, \max_c p(y=c|x) > \tau\}$$

where $\tau$ is a confidence threshold.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import LabelPropagation, LabelSpreading
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Create dataset
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)

# Simulate semi-supervised: only 10% labeled
n_labeled = 100
labeled_idx = np.random.choice(len(X), n_labeled, replace=False)
unlabeled_idx = np.setdiff1d(np.arange(len(X)), labeled_idx)

# Self-training from scratch
class SelfTrainer:
    def __init__(self, base_clf, threshold=0.9, max_iter=10):
        self.base_clf = base_clf
        self.threshold = threshold
        self.max_iter = max_iter
    
    def fit(self, X_labeled, y_labeled, X_unlabeled):
        X_train, y_train = X_labeled.copy(), y_labeled.copy()
        X_unlab = X_unlabeled.copy()
        
        for iteration in range(self.max_iter):
            self.base_clf.fit(X_train, y_train)
            if len(X_unlab) == 0:
                break
            proba = self.base_clf.predict_proba(X_unlab)
            max_proba = proba.max(axis=1)
            confident_idx = max_proba >= self.threshold
            if confident_idx.sum() == 0:
                break
            pseudo_labels = proba[confident_idx].argmax(axis=1)
            X_train = np.vstack([X_train, X_unlab[confident_idx]])
            y_train = np.concatenate([y_train, pseudo_labels])
            X_unlab = X_unlab[~confident_idx]
            print(f"Iter {iteration+1}: added {confident_idx.sum()} samples, {len(X_unlab)} unlabeled remain")
        return self
    
    def predict(self, X):
        return self.base_clf.predict(X)

# Baseline: supervised only
clf_supervised = LogisticRegression(max_iter=1000)
clf_supervised.fit(X[labeled_idx], y[labeled_idx])
acc_supervised = accuracy_score(y[unlabeled_idx], clf_supervised.predict(X[unlabeled_idx]))

# Self-training
self_trainer = SelfTrainer(LogisticRegression(max_iter=1000), threshold=0.85)
self_trainer.fit(X[labeled_idx], y[labeled_idx], X[unlabeled_idx])
acc_self = accuracy_score(y[unlabeled_idx], self_trainer.predict(X[unlabeled_idx]))

print(f"\nSupervised only accuracy: {acc_supervised:.4f}")
print(f"Self-training accuracy:   {acc_self:.4f}")

Iter 1: added 611 samples, 289 unlabeled remain
Iter 2: added 102 samples, 187 unlabeled remain
Iter 3: added 31 samples, 156 unlabeled remain
Iter 4: added 8 samples, 148 unlabeled remain
Iter 5: added 4 samples, 144 unlabeled remain
Iter 6: added 1 samples, 143 unlabeled remain
Iter 7: added 2 samples, 141 unlabeled remain
Iter 8: added 1 samples, 140 unlabeled remain
Iter 9: added 4 samples, 136 unlabeled remain
Iter 10: added 4 samples, 132 unlabeled remain

Supervised only accuracy: 0.8356
Self-training accuracy:   0.8344


## 2. Label Propagation

Graph-based method that propagates labels through a similarity graph.

### Algorithm
Build a graph where nodes are data points and edge weights reflect similarity:
$$W_{ij} = \exp\left(-\frac{||x_i - x_j||^2}{2\sigma^2}\right)$$

Normalize: $S = D^{-1/2}WD^{-1/2}$ where $D_{ii} = \sum_j W_{ij}$

Closed-form solution:
$$f = (I - \alpha S)^{-1}y$$

where $\alpha \in (0,1)$ controls propagation strength and $y$ contains known labels (0 for unlabeled).

## 3. Label Spreading

Similar to Label Propagation but uses normalized Laplacian and allows label correction:
$$f^{t+1} = \alpha S f^t + (1-\alpha) y$$

Converges to: $f^* = (I - \alpha S)^{-1}(1-\alpha)y$

- $\alpha$ closer to 1: more propagation (trust unlabeled structure)
- $\alpha$ closer to 0: less propagation (trust labeled data more)

In [2]:
# Label Propagation and Label Spreading
# sklearn uses -1 to denote unlabeled
y_semi = y.copy().astype(float)
y_semi[unlabeled_idx] = -1

# Label Propagation
lp = LabelPropagation(kernel='rbf', gamma=20, max_iter=1000)
lp.fit(X, y_semi)
acc_lp = accuracy_score(y[unlabeled_idx], lp.predict(X)[unlabeled_idx])

# Label Spreading
ls = LabelSpreading(kernel='rbf', alpha=0.2, max_iter=1000)
ls.fit(X, y_semi)
acc_ls = accuracy_score(y[unlabeled_idx], ls.predict(X)[unlabeled_idx])

print(f"Label Propagation accuracy: {acc_lp:.4f}")
print(f"Label Spreading accuracy:   {acc_ls:.4f}")

# Manual Label Propagation from scratch
def label_propagation_scratch(X, labeled_idx, y_labeled, alpha=0.5, sigma=1.0):
    n = len(X)
    # Build RBF similarity matrix
    from sklearn.metrics.pairwise import rbf_kernel
    W = rbf_kernel(X, gamma=1/(2*sigma**2))
    np.fill_diagonal(W, 0)
    # Row-normalize
    D = W.sum(axis=1)
    D_inv = np.diag(1.0 / (D + 1e-10))
    S = D_inv @ W
    # Initialize labels
    n_classes = len(np.unique(y_labeled))
    F = np.zeros((n, n_classes))
    for i, idx in enumerate(labeled_idx):
        F[idx, y_labeled[i]] = 1.0
    Y = F.copy()
    # Iterate
    for _ in range(100):
        F = alpha * S @ F + (1 - alpha) * Y
    return F.argmax(axis=1)

pred_scratch = label_propagation_scratch(X, labeled_idx, y[labeled_idx])
acc_scratch = accuracy_score(y[unlabeled_idx], pred_scratch[unlabeled_idx])
print(f"Label Propagation (scratch): {acc_scratch:.4f}")

Label Propagation accuracy: 0.7311
Label Spreading accuracy:   0.7378


Label Propagation (scratch): 0.7856


## 4. Co-Training

Requires two **independent views** (feature subsets) of the data:
1. Train classifiers $h_1$, $h_2$ on each view using labeled data
2. Each classifier labels the most confident unlabeled examples
3. Add those examples to the other classifier's training set
4. Repeat

**Assumption**: The two views are conditionally independent given the class label.

## 5. Pseudo-Labeling

Similar to self-training but used in deep learning:
$$\mathcal{L}_{total} = \mathcal{L}_{supervised} + \lambda(t) \mathcal{L}_{pseudo}$$

where $\lambda(t)$ is a ramp-up function that increases over training.

## 6. Mean Teacher (Consistency Regularization)

Two networks: **student** (trained normally) and **teacher** (exponential moving average of student weights):
$$\theta'_t = \alpha\theta'_{t-1} + (1-\alpha)\theta_t$$

Consistency loss:
$$\mathcal{L}_{cons} = ||f_{student}(x+\text{noise}) - f_{teacher}(x)||^2$$

The teacher provides stable pseudo-labels for unlabeled data.

## 7. FixMatch

State-of-the-art SSL combining pseudo-labeling + consistency regularization + strong augmentation:

$$\mathcal{L} = \mathcal{L}_s + \lambda \mathcal{L}_u$$

**Supervised loss** (on labeled data):
$$\mathcal{L}_s = \frac{1}{B}\sum_{b=1}^B H(y_b, p_m(y|\mathcal{A}(x_b)))$$

**Unsupervised loss** (on unlabeled data):
$$\mathcal{L}_u = \frac{1}{\mu B}\sum_{b=1}^{\mu B} \mathbf{1}[\max(q_b) \geq \tau] H(\hat{q}_b, p_m(y|\mathcal{A}(u_b)))$$

- $\mathcal{A}$: weak augmentation for pseudo-label generation
- $\mathcal{A}$: strong augmentation for consistency training  
- $\tau = 0.95$: confidence threshold
- $\hat{q}_b = \arg\max(q_b)$: pseudo-label from weakly-augmented image

## 8. MixMatch

Combines multiple techniques:
1. **Augmentation**: K augmentations of each unlabeled sample
2. **Sharpen** pseudo-labels: $\text{Sharpen}(p, T)_i = p_i^{1/T} / \sum_j p_j^{1/T}$ (low T = sharper)
3. **MixUp**: $\tilde{x} = \lambda x_1 + (1-\lambda)x_2$, $\tilde{y} = \lambda y_1 + (1-\lambda)y_2$

## 9. Semi-Supervised SVMs (S3VM / TSVM)

Extends SVM to exploit unlabeled data by pushing the decision boundary away from unlabeled points:
$$\min_{w,b,y^*} \frac{1}{2}||w||^2 + C\sum_i \xi_i + C^*\sum_j \xi_j^*$$
subject to: $y_i(w^Tx_i + b) \geq 1 - \xi_i$ and $|w^Tx_j^* + b| \geq 1 - \xi_j^*$

In [3]:
# Pseudo-labeling with deep learning style (sklearn simulation)
from sklearn.neural_network import MLPClassifier
from sklearn.base import clone

def pseudo_labeling(base_clf, X_l, y_l, X_u, threshold=0.9, n_iter=5):
    """Pseudo-labeling for semi-supervised learning."""
    clf = clone(base_clf)
    X_train, y_train = X_l.copy(), y_l.copy()
    X_unlabeled = X_u.copy()
    
    for i in range(n_iter):
        clf.fit(X_train, y_train)
        if len(X_unlabeled) == 0:
            break
        proba = clf.predict_proba(X_unlabeled)
        confidence = proba.max(axis=1)
        mask = confidence >= threshold
        if mask.sum() > 0:
            X_train = np.vstack([X_train, X_unlabeled[mask]])
            y_train = np.concatenate([y_train, proba[mask].argmax(axis=1)])
            X_unlabeled = X_unlabeled[~mask]
        print(f"Round {i+1}: {mask.sum()} samples added, {len(X_unlabeled)} remaining")
    return clf

mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
clf_pseudo = pseudo_labeling(mlp, X[labeled_idx], y[labeled_idx], X[unlabeled_idx])
acc_pseudo = accuracy_score(y[unlabeled_idx], clf_pseudo.predict(X[unlabeled_idx]))
print(f"\nPseudo-labeling accuracy: {acc_pseudo:.4f}")

# Co-training simulation (two feature views)
n_features = X.shape[1]
mid = n_features // 2
X_view1, X_view2 = X[:, :mid], X[:, mid:]

clf1 = LogisticRegression(max_iter=1000).fit(X_view1[labeled_idx], y[labeled_idx])
clf2 = LogisticRegression(max_iter=1000).fit(X_view2[labeled_idx], y[labeled_idx])

for _ in range(5):
    # clf1 labels for clf2
    p1 = clf1.predict_proba(X_view1[unlabeled_idx])
    conf1 = p1.max(axis=1)
    top_k1 = unlabeled_idx[np.argsort(conf1)[-20:]]
    
    # clf2 labels for clf1
    p2 = clf2.predict_proba(X_view2[unlabeled_idx])
    conf2 = p2.max(axis=1)
    top_k2 = unlabeled_idx[np.argsort(conf2)[-20:]]
    
    # Retrain
    add1_idx = np.concatenate([labeled_idx, top_k2])
    add2_idx = np.concatenate([labeled_idx, top_k1])
    clf1.fit(X_view1[add1_idx], y[add1_idx])
    clf2.fit(X_view2[add2_idx], y[add2_idx])

# Combine predictions
final_pred = (clf1.predict_proba(X_view1) + clf2.predict_proba(X_view2)).argmax(axis=1)
acc_cotrain = accuracy_score(y[unlabeled_idx], final_pred[unlabeled_idx])
print(f"Co-training accuracy: {acc_cotrain:.4f}")

# Summary
print("\n" + "="*50)
print("SUMMARY (10% labeled data)")
print("="*50)
print(f"Supervised only:     {acc_supervised:.4f}")
print(f"Self-training:       {acc_self:.4f}")
print(f"Label Propagation:   {acc_lp:.4f}")
print(f"Label Spreading:     {acc_ls:.4f}")
print(f"Pseudo-labeling:     {acc_pseudo:.4f}")
print(f"Co-training:         {acc_cotrain:.4f}")

Round 1: 723 samples added, 177 remaining


Round 2: 83 samples added, 94 remaining


Round 3: 25 samples added, 69 remaining


Round 4: 11 samples added, 58 remaining


Round 5: 3 samples added, 55 remaining

Pseudo-labeling accuracy: 0.8178


Co-training accuracy: 0.8589

SUMMARY (10% labeled data)
Supervised only:     0.8356
Self-training:       0.8344
Label Propagation:   0.7311
Label Spreading:     0.7378
Pseudo-labeling:     0.8178
Co-training:         0.8589


## When to Use Which Method?

| Method | Best For | Limitation |
|--------|----------|------------|
| Self-training | Quick baseline, tabular data | Error accumulation |
| Label Propagation | Small datasets, graph structure | Doesn't scale well |
| Co-training | Natural multi-view data | Requires independent views |
| Pseudo-labeling | Deep learning, image tasks | Threshold sensitivity |
| FixMatch | Image classification | Needs strong augmentations |
| Mean Teacher | Any deep learning task | Compute overhead |

---

## Additional Learning Resources

### Papers
- **FixMatch**: [https://arxiv.org/abs/2001.07685](https://arxiv.org/abs/2001.07685)
- **MixMatch**: [https://arxiv.org/abs/1905.02249](https://arxiv.org/abs/1905.02249)
- **Mean Teacher**: [https://arxiv.org/abs/1703.01780](https://arxiv.org/abs/1703.01780)
- **Label Propagation**: [https://citeseerx.ist.psu.edu/viewdoc/summary?doi=10.1.1.14.3864](https://citeseerx.ist.psu.edu/viewdoc/summary?doi=10.1.1.14.3864)
- **FlexMatch**: [https://arxiv.org/abs/2110.08263](https://arxiv.org/abs/2110.08263)

### Books
- **Semi-Supervised Learning** by Chapelle, Schölkopf, Zien: [https://mitpress.mit.edu/9780262514125/](https://mitpress.mit.edu/9780262514125/)

### Libraries
- **scikit-learn semi-supervised**: [https://scikit-learn.org/stable/modules/semi_supervised.html](https://scikit-learn.org/stable/modules/semi_supervised.html)
- **TorchSSL**: [https://github.com/TorchSSL/TorchSSL](https://github.com/TorchSSL/TorchSSL)
- **USB (Unified SSL Benchmark)**: [https://github.com/microsoft/Semi-supervised-learning](https://github.com/microsoft/Semi-supervised-learning)

### Courses
- Carnegie Mellon SSL Course: [http://www.cs.cmu.edu/~10715-f18/](http://www.cs.cmu.edu/~10715-f18/)